# 623 Stride v20: independent exact-PC LSTM

The byte-identical v19 input exposes only PC and current address. v20 trains h16 and h32 exact-PC LSTMs. Lossless causal same-input features include current/prior same-PC deltas and distinct-PC reuse distance. Natural gate/count heads and a generic rank-conditioned 255-exact-plus-OTHER delta head generate direct addresses. Every teacher rank is supervised without teacher or predicted action feedback. Decode is deterministic MAP; guard selects only the checkpoint, never a threshold. There is no normal tracker, candidate input, action template, page rule, request budget, probability threshold, or degree cap.

In [ ]:
import hashlib, math, os, pathlib, shutil, subprocess, sys, tarfile
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import userdata
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0),f'Select an A100 runtime, observed {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}'
torch.set_float32_matmul_precision('highest')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
torch.use_deterministic_algorithms(True)
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
import json
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/train_and_offline_infer.py'
CONTRACT_SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_stride/python/model_contract.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
assert MODEL_CONTRACT==json.loads(subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--describe-model-points'],text=True))
assert subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--field','run_id'],text=True).strip()==MODEL_CONTRACT['run_id']
assert subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--tags-csv'],text=True).strip()==','.join(p['model_tag'] for p in MODEL_CONTRACT['points'])
assert subprocess.check_output([sys.executable,CONTRACT_SCRIPT,'--base-tag'],text=True).strip()==MODEL_CONTRACT['points'][0]['model_tag']
RUN_ID=MODEL_CONTRACT['run_id']; DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_stride/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); archive=f'{DRIVE_ROOT}/{name}'
assert list(uploaded)==[name],f'Select exactly the single complete archive {name}'
pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle:
    members=handle.getmembers()
    for member in members:
        parts=pathlib.PurePosixPath(member.name)
        assert not parts.is_absolute() and '..' not in parts.parts,member.name
        assert not member.issym() and not member.islnk(),member.name
        assert member.isfile() or member.isdir(),member.name
    handle.extractall(INPUT_DIR,members=members)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,item=record.split(maxsplit=1); item=item.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified byte-identical v9 input',archive)

In [ ]:
import gzip, json
TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; ROLES=('train','guard','eval')
TRAINING=MODEL_CONTRACT['training_config']
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','candidates':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_candidates.csv.gz'} for role in ROLES}
for role,items in INPUTS.items():
    for path in items.values(): assert os.path.isfile(path),path
manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected={'status':'PASS','experiment_revision':MODEL_CONTRACT['experiment_revision'],'neural_role':'standalone_direct_action_prefetcher','source_decision_effective_external_input':['pc','addr'],'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True}
bad={k:(manifest.get(k),v) for k,v in expected.items() if manifest.get(k)!=v}; assert not bad,bad
assert manifest['training_runtime_fields']==['pc','addr']==manifest['inference_runtime_fields']
assert MODEL_CONTRACT['neural_role']=='standalone_direct_action_prefetcher' and MODEL_CONTRACT['delta_vocabulary_max_exact']==255 and MODEL_CONTRACT['decoder_previous_teacher_action_used_as_input'] is False

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[{'tag':point['model_tag'],'family':point['model_family'],'size':point['model_size'],'pair':point['architecture_pair_id'],'parameters':point['parameter_count']} for point in MODEL_CONTRACT['points']]
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-candidates',INPUTS[role]['candidates']]
 cmd += ['--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed',str(TRAINING['seed']),'--epochs',str(TRAINING['epochs']),'--chunk-len',str(TRAINING['chunk_len']),'--accumulate-chunks',str(TRAINING['accumulate_chunks']),'--learning-rate',str(TRAINING['learning_rate'])]
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm','model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],'parameter_count':spec['parameters'],'runtime_feature_count':MODEL_CONTRACT['runtime_feature_count'],'raw_runtime_feature_count':MODEL_CONTRACT['raw_runtime_feature_count'],'causal_runtime_feature_count':MODEL_CONTRACT['causal_runtime_feature_count'],'matched_normal_prefetcher':POLICY,'neural_role':'standalone_direct_action_prefetcher','same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':MODEL_CONTRACT['decoder_training_mode'],'decoder_previous_teacher_action_used_as_input':False,'decoder_previous_predicted_action_used_as_input':False,'all_teacher_ranks_supervised':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_templates_used_by_neural_inference':False,'probability_threshold_used':False,'threshold_related_hardcodes_used':False,'inference_policy_hardcodes_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'derived_features_use_teacher_or_future':False,'delta_vocabulary_source':'train_labels_only','delta_vocabulary_max_exact':255,'delta_other_escape':'signed_log_continuous_bounded_approximation','delta_other_decode_precision':'rounded_float32_approximate_except_exact_vocabulary','delta_coordinate_auxiliary_trained_on_all_teacher_actions':True,'delta_coordinate_used_for_decode_only_on_other':True,'full_signed_line_delta_range_reachable':False,'every_signed_line_delta_exactly_representable':False,'exact_delta_representability_scope':'train_vocabulary_only','gate_training_objective':MODEL_CONTRACT['gate_training_objective'],'positive_count_training_objective':MODEL_CONTRACT['positive_count_training_objective'],'delta_training_objective':MODEL_CONTRACT['delta_training_objective'],'deterministic_decoding':True,'stochastic_decoding':False,'decoder_sampling_roles':[],'sampled_outputs_used_as_decoder_feedback':False,'decode_per_callback_resource_watchdog':MODEL_CONTRACT['decode_per_callback_resource_watchdog'],'decode_per_role_resource_watchdog':MODEL_CONTRACT['decode_per_role_resource_watchdog'],'decode_resource_watchdog_is_neural_degree_cap':False,'successful_run_hit_decode_resource_watchdog':False,'checkpoint_selection':MODEL_CONTRACT['checkpoint_selection'],'guard_role':'checkpoint_selection_only_no_threshold_calibration','evaluation_used_for_checkpoint_selection':False,'evaluation_decode_passes':1,'experiment_revision':MODEL_CONTRACT['experiment_revision']}
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['training_config']==TRAINING and meta['training_config_pinned_by_run_id'] is True
 determinism=MODEL_CONTRACT['determinism_contract']; assert meta['training_device']=='cuda' and determinism['required_accelerator_name_contains'] in meta['training_device_name']
 assert meta['cublas_workspace_config']==determinism['cublas_workspace_config'] and meta['torch_deterministic_algorithms_enabled'] is True and meta['cudnn_deterministic'] is True and meta['cudnn_benchmark'] is False and meta['float32_matmul_precision']==determinism['float32_matmul_precision']
 source_paths={'trainer_source_sha256':SCRIPT,'model_contract_source_sha256':CONTRACT_SCRIPT,'threshold_free_policy_source_sha256':f'{REPO}/formal_NN_training/common/threshold_free_policy.py'}
 for key,path in source_paths.items(): assert meta[key]==hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest(),(key,meta[key],path)
 assert meta['training_runtime_fields']==['pc','addr']==meta['inference_runtime_fields']
 assert 0<meta['delta_vocabulary_exact_size']<=255 and len(meta['delta_vocabulary_exact'])==meta['delta_vocabulary_exact_size']
 assert 1<=meta['selected_guard_epoch']<=TRAINING['epochs'] and meta['checkpoint_selection_roles']==['guard']
 prior=meta['gate_empirical_prior']; bias=meta['gate_initial_bias']; assert len(prior)==len(bias)==2 and math.isclose(sum(prior),1.0) and all(math.isclose(math.log(p),b,rel_tol=1e-6,abs_tol=1e-7) for p,b in zip(prior,bias))
 count_stats=meta['request_count_training_label_statistics']; expected_count_bias=sum(int(v)*math.log(int(k)) for k,v in count_stats['count_distribution'].items() if int(k)>0)/count_stats['positive_callbacks']; assert math.isclose(meta['positive_log_count_initial_bias'],expected_count_bias,rel_tol=1e-6,abs_tol=1e-7)
 vocab_stats=meta['delta_vocabulary_statistics']['train']; class_counts=[0]*256
 for i,value in enumerate(meta['delta_vocabulary_train_frequencies']): class_counts[i]=value
 class_counts[255]=vocab_stats['other_escape_actions']; denominator=vocab_stats['teacher_actions']+256; expected_prior=[(value+1)/denominator for value in class_counts]
 assert all(math.isclose(a,b,rel_tol=1e-6,abs_tol=1e-9) for a,b in zip(meta['delta_class_empirical_prior'],expected_prior)) and all(math.isclose(a,math.log(b),rel_tol=1e-6,abs_tol=1e-7) for a,b in zip(meta['delta_class_initial_bias'],expected_prior))
 encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}; assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','selected_guard_epoch','decision_rule','offline_normal_entries','offline_nn_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':MODEL_CONTRACT['experiment_revision'],'model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the v20 output archive to the matching server run and launch replay. Server validation requires the independent exact-PC encoder, train-only delta vocabulary, continuous OTHER escape, full rank supervision without action feedback, deterministic guard-selected checkpoint, and matching input hashes.